# 01 Mesh preprocessing

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from stl import mesh
from os.path import exists, split, join
from os import makedirs

from base.utils import find_neighbours_with_knn, load_ply, list_all_plymeshes, return_rings
from base.LBC_centreline import return_LBC_centreline
from base.refine_centreline import refine


C:\Users\tangr\AppData\Roaming\Python\Python310\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


ImportError: DLL load failed while importing utility_random_walk: The specified module could not be found.

The first step is to generate a listing of all ply mesh filepaths from a directory. 

In [ ]:
# change here the path to the repository of karst catalogue
data_repository = "F:/ScanLeica/data_paper/KarstConduitCatalogue" 
passages_filepaths = list_all_plymeshes(data_repository)

In [ ]:
passages_filepaths

In [ ]:
idx = 19

In [ ]:
plymesh_filepath =  passages_filepaths[idx]
stlmesh_filepath = plymesh_filepath[:-3]+"stl"

The first step is to use a stl.mesh.Mesh() object to do our calculations with. 

In [ ]:
keepfile = True # change to false to regenerate a clean stl mesh file.

if exists(stlmesh_filepath) and keepfile==True:
    # if there exists already a file in stl format.
    mesh_data = mesh.Mesh.from_file(stlmesh_filepath)
else:
    mesh_data = load_ply(plymesh_filepath)
    mesh_data.save(stlmesh_filepath)

Now that we have a mesh object, we can query it easily. In numpy-stl.mesh.Mesh() objects, the surface is defined by the .vectors attribute, an array of triangle vertex coordinates whose shape is the following (ntriangles, 3, 3). By calling the method .update_normals() and .update_centroids(), we can recompute the face normals and their centroids, storing them in separate attributes.

In [ ]:
# not strictly necessary at this stage, but can be done.
update = False

if update:
    mesh_data.update_centroids() 
    mesh_data.update_normals()

One thing we can do is to extract the mesh faces centroids and plot them as a point cloud.

In [ ]:
fig = plt.figure() 

ax = fig.add_subplot(projection="3d")

ax.scatter(*mesh_data.centroids[::100, [0, 1, 2]].T, s=0.5, c = mesh_data.vectors[::100, 1, 2], cmap = "summer")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_zlabel("z [m]")

plt.show()

Another preprocessing step we might have to perform is the following: to compute a matrix of triangle adjacency from the mesh. This is calculated once and can be reused for the intersection computations. 

In [ ]:
keep_triangles = True
adj_triangles_filepath = stlmesh_filepath[:-4] + "_adjacent_triangles.txt"

if exists(adj_triangles_filepath) and keep_triangles == True:
    adjacent_triangles = np.loadtxt(adj_triangles_filepath)
else:
    adjacent_triangles = find_neighbours_with_knn(mesh_data)
    np.savetxt  
    (adj_triangles_filepath, adjacent_triangles)

Finally, we can generate or reuse an existing centreline.

In [ ]:
centreline_directory = join(split(passages_filepaths[idx])[0][:-5], "geometric_descriptors")

if exists(centreline_directory):
    print("centreline directory exists already.")
else:
    makedirs(centreline_directory)

centreline_filepath = join(centreline_directory, split(passages_filepaths[idx])[1][:-12]+"centreline.txt")

In [ ]:
keep_centreline = True
refine_centreline = True

if exists(centreline_filepath) and keep_centreline:
    centreline = np.loadtxt(centreline_filepath, skiprows=1)
    
    if refine_centreline:
        distances = np.linalg.norm(np.diff(centreline, axis = 0), axis = 1)

        # by how much to refine. 
        while np.max(distances) > 1.5 * np.median(distances):
            centreline  = refine(centreline, 1)
            distances = np.linalg.norm(np.diff(centreline, axis = 0), axis = 1)
        np.savetxt(centreline_filepath, centreline, header="X, Y, Z")

else:
    lbc_args = dict(init_attraction = 4, 
                    init_contraction = 0.5, 
                    down_sample = 0.7)

    ct_args = dict(centreline_down_sample= 0.7, 
                   keep_adjacent_triangles=True)

    LBC_result = return_LBC_centreline(stlmesh_filepath, 
                                    adj_triangles_filepath, 
                                    lbc_args=lbc_args, 
                                    ct_args=ct_args)

    centreline = LBC_result["centreline"]
    
    if refine_centreline:
        distances = np.linalg.norm(np.diff(centreline, axis = 0), axis = 1)

        # by how much to refine. 
        while np.max(distances) > 1.5 * np.median(distances):
            centreline  = refine(centreline, 1)
            distances = np.linalg.norm(np.diff(centreline, axis = 0), axis = 1)
        
    np.savetxt(centreline_filepath, centreline, header="X, Y, Z")

Find the rings of the mesh as these correspond to potential inlets and outlets.

In [ ]:
rings = return_rings(mesh_data, adjacent_triangles)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(*mesh_data[::100, [0,1]].T, color = "lightgrey", s = 2)

ax.plot(*centreline[:, [0,1]].T, zorder = 100, lw = 2, label = "centreline")
for key, value in rings.items():
    if "ring" in key:
        ax.scatter(*value[:, [0,1]].T, color="C1", s=5)
        
ax.set_aspect("equal")
ax.legend()
ax.set_ylabel("Y [m]")
ax.set_xlabel("X [m]")

In [ ]:
# now we put everything in one big loop to process all the datasets. 
